# FastAPI Uber Ride Fare Prediction Pipeline

This notebook prepares the deployment pipeline for the Uber fare prediction model. FastAPI is used because this project needs a production-style prediction endpoint, input validation, Swagger documentation, and reusable backend logic. Streamlit can still be added later as a demo UI, but FastAPI is the stronger deployment choice for serving a model.

### Deployment Goal

The API user should enter only raw trip values from the original dataset, excluding `Unnamed: 0`, `key`, and `fare_amount`. The app automatically calculates all engineered features before predicting fare.

#### Raw Input Fields Expected

- `pickup_datetime`
- `pickup_longitude`
- `pickup_latitude`
- `dropoff_longitude`
- `dropoff_latitude`
- `passenger_count`

### Project Folder

In [ ]:
from pathlib import Path

API_DIR = Path(r'C:\Eni data\PYTHON\PYTHON CODES & PROJECTS\ride_fare_fastapi')
print(API_DIR)
print('Exists:', API_DIR.exists())

### Step 1: Install Requirements

Run this once if dependencies are missing.

In [ ]:
# Run this cell only if required.
# %pip install -r "C:/Eni data/PYTHON/PYTHON CODES & PROJECTS/ride_fare_fastapi/requirements.txt"

### Step 2: Train Final Model and Export Artifacts

This script repeats the deployment-safe preprocessing, trains XGBoost, and saves the model plus preprocessing artifacts. The saved artifacts include feature column order, train-derived peak hours, and pickup/dropoff KMeans models.

In [ ]:
import subprocess

result = subprocess.run(
    ['python', 'train_and_export.py'],
    cwd=API_DIR,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Training/export failed')

### Step 3: Inspect Saved Artifacts

In [ ]:
artifacts_dir = API_DIR / 'artifacts'
for path in artifacts_dir.glob('*'):
    print(path.name, path.stat().st_size, 'bytes')

### Step 4: Start FastAPI Server

Run this command in a terminal. It is not started inside the notebook because the server keeps running until manually stopped.

In [ ]:
print(f'cd "{API_DIR}"')
print('uvicorn app.main:app --reload')
print('Swagger UI: http://127.0.0.1:8000/docs')

### Step 5: Example Request

The request contains only raw user-facing fields. Time conversion, distance, airport flag, peak-hour flag, clusters, and encoded features are calculated by the API.

In [ ]:
sample_request = {
    'pickup_datetime': '2015-05-07 19:52:06 UTC',
    'pickup_longitude': -73.9855,
    'pickup_latitude': 40.758,
    'dropoff_longitude': -73.9712,
    'dropoff_latitude': 40.7831,
    'passenger_count': 1
}
sample_request

### Step 6: Test the API After Server Starts

In [ ]:
# Run this only after starting the FastAPI server.
# import requests
# response = requests.post('http://127.0.0.1:8000/predict', json=sample_request)
# print(response.status_code)
# print(response.json())

### Deployment Conclusion

This deployment pipeline is designed so the model-serving layer matches the notebook logic. Users provide raw trip inputs only, while the application calculates all derived features consistently at inference time. This is important because production models should not expect users to know engineered columns like `distance_km`, `airport_trip`, `is_peak_hour`, or cluster labels. FastAPI is the right first deployment choice because it provides input validation, a clear `/predict` endpoint, automatic Swagger documentation, and a backend-ready structure. A Streamlit interface can be added later on top of this API if a more visual demo is needed.